<a href="https://www.kaggle.com/code/joanaowusuappiah/alzheimer-asl-perfusion?scriptVersionId=292254888" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Synthetic ASL Perfusion Analysis

**Author: Joana Owusu-Appiah.** A methodology demonstration using 80 simulated subjects and 24 regional cerebral-blood-flow features. No patient scans are analysed. Group differences are built into the simulation, so high classification scores do not establish clinical performance.

The workflow compares groups, visualizes simulated effects and evaluates logistic regression with fold-isolated preprocessing. Optional atlas figures are illustrative, not subject-specific MRI measurements.

---
## 1. Environment Setup

In [ ]:
# Install requirements.txt before running. Optional atlas packages are in requirements-atlas.txt.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Neuroimaging
import os
ENABLE_ATLAS = os.environ.get("ASL_ENABLE_ATLAS", "0") == "1"

# Statistics & ML
from scipy import stats
from sklearn.metrics import roc_curve, auc, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, cross_val_predict, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# Plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Output directory
OUTPUT_DIR = Path('./outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

print("Environment ready!")

---
## 2. Create Preclinical AD Dataset

We simulate a dataset matching the Dallas Lifespan Brain Study structure, with:
- Cognitively normal older adults (age 55-85)
- Amyloid PET status (SUVR values)
- Regional CBF values from ASL-MRI

CBF patterns are based on published literature (Binnewijzend 2013, Mattsson 2014, Michels 2016).

In [ ]:
def create_preclinical_ad_dataset(n_subjects=80, seed=42):
    """
    Create dataset comparing amyloid-positive vs amyloid-negative 
    cognitively normal adults.
    
    Based on:
    - ~30% of cognitively normal elderly are amyloid-positive
    - Aβ+ show 5-15% hypoperfusion in AD-signature regions
    - Effect sizes from Binnewijzend 2013, Mattsson 2014
    """
    np.random.seed(seed)
    
    # Age range: older adults where amyloid status is clinically meaningful
    ages = np.random.uniform(55, 85, n_subjects)
    
    # Amyloid status: ~35% positive (higher in older samples)
    amyloid_positive = np.random.choice([True, False], n_subjects, p=[0.35, 0.65])
    
    # Brain regions with realistic CBF values and amyloid effects
    # Format: base CBF, age effect (per year from 60), amyloid effect
    regions = {
        # AD-SIGNATURE REGIONS (large amyloid effect)
        'PosteriorCingulate_L': {'base': 58, 'age': -0.35, 'amyloid': -9.5},
        'PosteriorCingulate_R': {'base': 58, 'age': -0.35, 'amyloid': -9.5},
        'Precuneus_L': {'base': 54, 'age': -0.30, 'amyloid': -8.5},
        'Precuneus_R': {'base': 54, 'age': -0.30, 'amyloid': -8.5},
        'Hippocampus_L': {'base': 42, 'age': -0.28, 'amyloid': -7.0},
        'Hippocampus_R': {'base': 42, 'age': -0.28, 'amyloid': -7.0},
        'ParaHippocampal_L': {'base': 40, 'age': -0.25, 'amyloid': -6.5},
        'ParaHippocampal_R': {'base': 40, 'age': -0.25, 'amyloid': -6.5},
        'TemporalMed_L': {'base': 44, 'age': -0.28, 'amyloid': -6.0},
        'TemporalMed_R': {'base': 44, 'age': -0.28, 'amyloid': -6.0},
        
        # MODERATE EFFECT REGIONS
        'TemporalLat_L': {'base': 48, 'age': -0.25, 'amyloid': -4.0},
        'TemporalLat_R': {'base': 48, 'age': -0.25, 'amyloid': -4.0},
        'Parietal_L': {'base': 50, 'age': -0.22, 'amyloid': -3.5},
        'Parietal_R': {'base': 50, 'age': -0.22, 'amyloid': -3.5},
        'AnteriorCingulate_L': {'base': 52, 'age': -0.20, 'amyloid': -2.5},
        'AnteriorCingulate_R': {'base': 52, 'age': -0.20, 'amyloid': -2.5},
        
        # CONTROL REGIONS (minimal/no amyloid effect)
        'Frontal_L': {'base': 52, 'age': -0.30, 'amyloid': -1.0},
        'Frontal_R': {'base': 52, 'age': -0.30, 'amyloid': -1.0},
        'Occipital_L': {'base': 55, 'age': -0.18, 'amyloid': 0.0},
        'Occipital_R': {'base': 55, 'age': -0.18, 'amyloid': 0.0},
        'Motor_L': {'base': 48, 'age': -0.15, 'amyloid': 0.0},
        'Motor_R': {'base': 48, 'age': -0.15, 'amyloid': 0.0},
        'Cerebellum_L': {'base': 45, 'age': -0.12, 'amyloid': 0.0},
        'Cerebellum_R': {'base': 45, 'age': -0.12, 'amyloid': 0.0},
    }
    
    data = []
    for i, (age, is_amyloid_pos) in enumerate(zip(ages, amyloid_positive)):
        # Demographics
        row = {
            'subject_id': f'sub-{i+1:03d}',
            'age': age,
            'sex': np.random.choice(['M', 'F']),
            'education_years': np.random.randint(12, 21),
            'amyloid_status': 'Positive' if is_amyloid_pos else 'Negative',
            # Amyloid PET SUVR (standardized uptake value ratio)
            'amyloid_suvr': np.random.normal(1.85, 0.25) if is_amyloid_pos else np.random.normal(1.08, 0.12),
            # Cognitive scores - all NORMAL (this is key!)
            'mmse': np.random.randint(27, 31),
            'moca': np.random.randint(26, 31),
            'cdr': 0,  # Clinical Dementia Rating = 0 (no dementia)
        }
        
        # Generate CBF for each region
        for region, params in regions.items():
            cbf = params['base']
            cbf += params['age'] * (age - 60)  # Age effect
            if is_amyloid_pos:
                cbf += params['amyloid']  # Amyloid effect
            cbf += np.random.normal(0, 2.5)  # Individual variation
            cbf = max(cbf, 15)  # Physiological floor
            row[region] = cbf
        
        data.append(row)
    
    return pd.DataFrame(data), list(regions.keys())

# Create dataset
df, region_cols = create_preclinical_ad_dataset(n_subjects=80)

# Summary
n_pos = (df['amyloid_status'] == 'Positive').sum()
n_neg = (df['amyloid_status'] == 'Negative').sum()

print("="*60)
print("DATASET SUMMARY")
print("="*60)
print(f"Total subjects: {len(df)}")
print(f"  Amyloid-Positive: {n_pos} ({100*n_pos/len(df):.0f}%)")
print(f"  Amyloid-Negative: {n_neg} ({100*n_neg/len(df):.0f}%)")
print(f"\nAge: {df['age'].mean():.1f} ± {df['age'].std():.1f} years")
print(f"MMSE: {df['mmse'].mean():.1f} ± {df['mmse'].std():.1f} (all normal)")
print(f"\nBrain regions: {len(region_cols)}")

In [ ]:
# Preview data
print("Sample data:")
display(df.head(10))

In [ ]:
# Verify groups are matched on age and cognition
print("Group Comparison (describing simulated groups):")
print("="*50)

for var in ['age', 'mmse', 'education_years']:
    pos_mean = df[df['amyloid_status']=='Positive'][var].mean()
    neg_mean = df[df['amyloid_status']=='Negative'][var].mean()
    t, p = stats.ttest_ind(
        df[df['amyloid_status']=='Positive'][var],
        df[df['amyloid_status']=='Negative'][var]
    )
    print(f"{var:20s}: Aβ- = {neg_mean:.1f}, Aβ+ = {pos_mean:.1f}, p = {p:.3f}")

print("\nNonsignificance alone does not establish group equivalence.")

---
## 3. Statistical Analysis: CBF Differences by Amyloid Status

In [ ]:
# Compare CBF between amyloid groups for each region
pos_df = df[df['amyloid_status'] == 'Positive']
neg_df = df[df['amyloid_status'] == 'Negative']

results = []
for region in region_cols:
    # T-test
    t_stat, p_val = stats.ttest_ind(neg_df[region], pos_df[region], equal_var=False)
    
    # Effect size (Cohen's d)
    pooled_std = np.sqrt(((len(neg_df)-1) * neg_df[region].var() + (len(pos_df)-1) * pos_df[region].var()) / (len(neg_df) + len(pos_df) - 2))
    cohens_d = (neg_df[region].mean() - pos_df[region].mean()) / pooled_std
    
    # Percent reduction
    pct_reduction = 100 * (neg_df[region].mean() - pos_df[region].mean()) / neg_df[region].mean()
    
    results.append({
        'Region': region,
        'CBF_Aβ-': neg_df[region].mean(),
        'CBF_Aβ+': pos_df[region].mean(),
        'Difference': neg_df[region].mean() - pos_df[region].mean(),
        'Pct_Reduction': pct_reduction,
        't_statistic': t_stat,
        'p_value': p_val,
        'Cohens_d': cohens_d,
        'Sig': '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else ''
    })

stats_df = pd.DataFrame(results).sort_values('p_value')

print("CBF COMPARISON: Amyloid-Negative vs Amyloid-Positive")
print("(Positive values = hypoperfusion in Aβ+ group)")
print("="*80)
display(stats_df.round(3))

In [ ]:
# Summary of significant findings
sig_regions = stats_df[stats_df['p_value'] < 0.05]

print(f"\n🎯 SIGNIFICANT FINDINGS ({len(sig_regions)}/{len(region_cols)} regions)")
print("="*60)
print("\nRegions with significant hypoperfusion in Aβ+ subjects:")
for _, row in sig_regions.iterrows():
    print(f"  • {row['Region']:25s}: {row['Pct_Reduction']:5.1f}% ↓  (d={row['Cohens_d']:.2f}, p={row['p_value']:.4f})")

In [ ]:
# Correlation: Amyloid burden (SUVR) vs regional CBF
print("\nCORRELATION: Amyloid SUVR vs Regional CBF")
print("="*60)

corr_results = []
for region in region_cols:
    r, p = stats.pearsonr(df['amyloid_suvr'], df[region])
    corr_results.append({'Region': region, 'r': r, 'p': p})

corr_df = pd.DataFrame(corr_results).sort_values('r')
print("\nStrongest negative correlations (higher amyloid → lower CBF):")
for _, row in corr_df.head(6).iterrows():
    sig = '***' if row['p'] < 0.001 else '**' if row['p'] < 0.01 else '*' if row['p'] < 0.05 else ''
    print(f"  {row['Region']:25s}: r = {row['r']:.3f} {sig}")

---
## 4. Visualization

In [ ]:
# Figure 1: CBF by amyloid status in AD-signature regions
fig, axes = plt.subplots(2, 3, figsize=(14, 9))

ad_signature = ['PosteriorCingulate_L', 'Precuneus_L', 'Hippocampus_L',
                'ParaHippocampal_L', 'TemporalMed_L', 'Parietal_L']

for ax, region in zip(axes.flat, ad_signature):
    # Box plot
    sns.boxplot(data=df, x='amyloid_status', y=region,
                palette={'Negative': '#27ae60', 'Positive': '#e74c3c'},
                ax=ax, width=0.5, order=['Negative', 'Positive'])
    
    # Individual points
    sns.stripplot(data=df, x='amyloid_status', y=region,
                  color='black', alpha=0.4, size=4, ax=ax, order=['Negative', 'Positive'])
    
    # Stats annotation
    row = stats_df[stats_df['Region'] == region].iloc[0]
    ax.set_title(f"{region.replace('_', ' ')}\n{row['Pct_Reduction']:.1f}% reduction, p={row['p_value']:.4f} {row['Sig']}",
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('CBF (ml/100g/min)')
    ax.set_xticklabels(['Aβ-', 'Aβ+'])

plt.suptitle('Simulated Cerebral Blood Flow by Amyloid Status\n(Synthetic data; imposed group differences)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig1_cbf_by_amyloid.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 2: Effect size plot
fig, ax = plt.subplots(figsize=(10, 8))

plot_data = stats_df.sort_values('Cohens_d', ascending=True)
colors = ['#e74c3c' if p < 0.05 else '#bdc3c7' for p in plot_data['p_value']]

bars = ax.barh(range(len(plot_data)), plot_data['Cohens_d'], color=colors, edgecolor='white')
ax.set_yticks(range(len(plot_data)))
ax.set_yticklabels([r.replace('_', ' ') for r in plot_data['Region']], fontsize=10)
ax.set_xlabel("Cohen's d (Effect Size)", fontsize=12)
ax.set_title('CBF Reduction in Amyloid-Positive vs Negative\n(Red = statistically significant)',
             fontsize=13, fontweight='bold')

# Reference lines
ax.axvline(0.5, color='orange', linestyle='--', alpha=0.7, label='Medium effect (0.5)')
ax.axvline(0.8, color='red', linestyle='--', alpha=0.7, label='Large effect (0.8)')
ax.axvline(0, color='black', linewidth=0.5)
ax.legend(loc='lower right')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig2_effect_sizes.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 3: Amyloid burden vs CBF scatter plots
fig, axes = plt.subplots(2, 3, figsize=(14, 9))

for ax, region in zip(axes.flat, ad_signature):
    colors = df['amyloid_status'].map({'Negative': '#27ae60', 'Positive': '#e74c3c'})
    ax.scatter(df['amyloid_suvr'], df[region], c=colors, alpha=0.6, s=50, edgecolor='white')
    
    # Regression line
    z = np.polyfit(df['amyloid_suvr'], df[region], 1)
    x_line = np.linspace(df['amyloid_suvr'].min(), df['amyloid_suvr'].max(), 100)
    ax.plot(x_line, np.poly1d(z)(x_line), 'k--', linewidth=2)
    
    # Correlation
    r, p = stats.pearsonr(df['amyloid_suvr'], df[region])
    ax.set_title(f"{region.replace('_', ' ')}\nr = {r:.2f}, p = {p:.4f}", fontsize=11, fontweight='bold')
    ax.set_xlabel('Amyloid SUVR')
    ax.set_ylabel('CBF (ml/100g/min)')

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#27ae60', label='Aβ-'),
                   Patch(facecolor='#e74c3c', label='Aβ+')]
fig.legend(handles=legend_elements, loc='upper center', ncol=2, bbox_to_anchor=(0.5, 0.02))

plt.suptitle('Relationship Between Amyloid Burden and Regional Perfusion',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig3_amyloid_cbf_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Optional atlas illustration

Disabled by default to keep the core analysis offline. Enable with `ASL_ENABLE_ATLAS=1` and install `requirements-atlas.txt`. Cortical labels are approximate bilateral illustrations; hippocampal and cerebellar features are not mapped. These figures are not reconstructions of patient scans.

In [ ]:
# Create brain map of CBF reduction

try:
    if not ENABLE_ATLAS:
        raise RuntimeError("optional atlas disabled; set ASL_ENABLE_ATLAS=1 to enable")
    import nibabel as nib
    from nilearn import plotting, datasets, surface
    # Harvard-Oxford atlas
    atlas = datasets.fetch_atlas_harvard_oxford('cort-maxprob-thr25-2mm')
    
    # atlas.maps is already a NIfTI image object in newer nilearn
    if isinstance(atlas.maps, str):
        atlas_img = nib.load(atlas.maps)
    else:
        atlas_img = atlas.maps  # Already a Nifti1Image
    
    atlas_data = atlas_img.get_fdata()
    atlas_labels = atlas.labels
    
    print(f"Atlas loaded: {len(atlas_labels)} regions")
    print(f"Atlas shape: {atlas_data.shape}")
    
    # Map our regions to Harvard-Oxford labels
    region_mapping = {
        'PosteriorCingulate_L': 'Cingulate Gyrus, posterior division',
        'PosteriorCingulate_R': 'Cingulate Gyrus, posterior division',
        'Precuneus_L': 'Precuneous Cortex',
        'Precuneus_R': 'Precuneous Cortex',
        'ParaHippocampal_L': 'Parahippocampal Gyrus, anterior division',
        'ParaHippocampal_R': 'Parahippocampal Gyrus, anterior division',
        'TemporalMed_L': 'Middle Temporal Gyrus, posterior division',
        'TemporalMed_R': 'Middle Temporal Gyrus, posterior division',
        'TemporalLat_L': 'Superior Temporal Gyrus, posterior division',
        'TemporalLat_R': 'Superior Temporal Gyrus, posterior division',
        'Parietal_L': 'Angular Gyrus',
        'Parietal_R': 'Angular Gyrus',
        'Frontal_L': 'Superior Frontal Gyrus',
        'Frontal_R': 'Superior Frontal Gyrus',
        'Occipital_L': 'Lateral Occipital Cortex, superior division',
        'Occipital_R': 'Lateral Occipital Cortex, superior division',
        'Motor_L': 'Precentral Gyrus',
        'Motor_R': 'Precentral Gyrus',
        'AnteriorCingulate_L': 'Cingulate Gyrus, anterior division',
        'AnteriorCingulate_R': 'Cingulate Gyrus, anterior division',
    }
    
    # Create reduction map
    reduction_map = np.zeros_like(atlas_data, dtype=float)
    
    # This atlas is bilateral. Average mapped left/right features rather than
    # overwriting the same atlas label with the right hemisphere value.
    for ho_label in set(region_mapping.values()):
        source_regions = [r for r, label in region_mapping.items() if label == ho_label]
        reduction = stats_df.loc[stats_df['Region'].isin(source_regions), 'Pct_Reduction'].mean()
        for i, label in enumerate(atlas_labels):
            if ho_label.lower() == label.lower():
                reduction_map[atlas_data == i] = reduction
                break

    reduction_img = nib.Nifti1Image(reduction_map, atlas_img.affine)
    BRAIN_VIS_AVAILABLE = True
    
    print(f"✓ Brain map created")
    print(f"  Regions mapped: {(reduction_map > 0).sum()} voxels")

except Exception as e:
    print(f"⚠ Atlas processing failed: {type(e).__name__}: {e}")
    print("  Brain slice visualizations will be skipped")
    BRAIN_VIS_AVAILABLE = False

In [ ]:
# Figure 4: Brain slices showing CBF reduction pattern
if BRAIN_VIS_AVAILABLE:
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))

    # Axial slices
    plotting.plot_stat_map(
        reduction_img,
        display_mode='z',
        cut_coords=[-25, -5, 15, 35],
        cmap='hot',
        colorbar=True,
        title='CBF Reduction (%) in Amyloid-Positive vs Negative - Axial View',
        axes=axes[0]
    )

    # Sagittal slices
    plotting.plot_stat_map(
        reduction_img,
        display_mode='x',
        cut_coords=[-45, -20, 0, 20, 45],
        cmap='hot',
        colorbar=True,
        title='Sagittal View',
        axes=axes[1]
    )

    plt.savefig(OUTPUT_DIR / 'fig4_brain_slices.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Skipping brain slice visualization (atlas not available)")

In [ ]:
# Figure 5: Glass brain visualization
if BRAIN_VIS_AVAILABLE:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    for ax, view in zip(axes, ['x', 'y', 'z']):
        plotting.plot_glass_brain(
            reduction_img,
            display_mode=view,
            cmap='hot',
            colorbar=True,
            title=f'Glass Brain ({view.upper()} projection)',
            axes=ax
        )

    plt.suptitle('Spatial Pattern of CBF Reduction in Preclinical AD',
                 fontsize=13, fontweight='bold', y=1.05)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'fig5_glass_brain.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Skipping glass brain visualization (atlas not available)")

In [ ]:
# Figure 6: Surface rendering
if BRAIN_VIS_AVAILABLE:
    try:
        fsaverage = datasets.fetch_surf_fsaverage()

        # Project to surface
        texture_left = surface.vol_to_surf(reduction_img, fsaverage.pial_left)
        texture_right = surface.vol_to_surf(reduction_img, fsaverage.pial_right)

        fig, axes = plt.subplots(2, 2, figsize=(12, 10), subplot_kw={'projection': '3d'})

        plotting.plot_surf_stat_map(fsaverage.infl_left, texture_left,
            hemi='left', view='lateral', cmap='hot', colorbar=False,
            title='Left Lateral', axes=axes[0, 0])

        plotting.plot_surf_stat_map(fsaverage.infl_left, texture_left,
            hemi='left', view='medial', cmap='hot', colorbar=False,
            title='Left Medial', axes=axes[0, 1])

        plotting.plot_surf_stat_map(fsaverage.infl_right, texture_right,
            hemi='right', view='lateral', cmap='hot', colorbar=False,
            title='Right Lateral', axes=axes[1, 0])

        plotting.plot_surf_stat_map(fsaverage.infl_right, texture_right,
            hemi='right', view='medial', cmap='hot', colorbar=True,
            title='Right Medial', axes=axes[1, 1])

        plt.suptitle('Cortical Surface: CBF Reduction Pattern (%) in Preclinical AD',
                     fontsize=13, fontweight='bold', y=0.98)
        plt.savefig(OUTPUT_DIR / 'fig6_surface_map.png', dpi=150, bbox_inches='tight')
        plt.show()
    except Exception as e:
        print(f"Surface rendering failed: {e}")
else:
    print("Skipping surface visualization (atlas not available)")

---
## 6. Classification: Can CBF Predict Amyloid Status?

In [ ]:
X = df[region_cols].values
y = (df['amyloid_status'] == 'Positive').astype(int).values
# Each fold fits its own scaler using only the training subjects.
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(clf, X, y, cv=cv, scoring='roc_auc')
y_prob = cross_val_predict(clf, X, y, cv=cv, method='predict_proba')[:, 1]
print(f'Synthetic five-fold AUC: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')


In [ ]:
# Figure 7: ROC curve
clf.fit(X, y)  # Full-data fit is used only for descriptive coefficients.

fpr, tpr, thresholds = roc_curve(y, y_prob)
roc_auc = auc(fpr, tpr)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC curve
axes[0].plot(fpr, tpr, color='#e74c3c', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[0].set_xlabel('False Positive Rate', fontsize=12)
axes[0].set_ylabel('True Positive Rate', fontsize=12)
axes[0].set_title('Synthetic out-of-fold ROC', fontsize=13, fontweight='bold')
axes[0].legend(loc='lower right', fontsize=11)
axes[0].set_xlim([0, 1])
axes[0].set_ylim([0, 1.05])

# Feature importance
importance = pd.DataFrame({
    'Region': region_cols,
    'Coefficient': np.abs(clf.named_steps['logisticregression'].coef_[0])
}).sort_values('Coefficient', ascending=True)

colors = ['#e74c3c' if r in ad_signature else '#3498db' for r in importance['Region']]
axes[1].barh(range(len(importance)), importance['Coefficient'], color=colors)
axes[1].set_yticks(range(len(importance)))
axes[1].set_yticklabels([r.replace('_', ' ') for r in importance['Region']], fontsize=9)
axes[1].set_xlabel('|Coefficient|', fontsize=12)
axes[1].set_title('Full-data coefficients (descriptive only)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig7_classification.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Confusion matrix
y_pred = (y_prob >= 0.5).astype(int)

print("\nClassification Report:")
print(classification_report(y, y_pred, target_names=['Aβ-', 'Aβ+']))

cm = confusion_matrix(y, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Aβ-', 'Aβ+'], yticklabels=['Aβ-', 'Aβ+'])
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Synthetic out-of-fold confusion matrix', fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig8_confusion_matrix.png', dpi=150)
plt.show()

---
## 7. Summary and Conclusions

In [ ]:
print(f"Synthetic cohort: {len(df)} subjects, {len(region_cols)} regional features")
print(f"Out-of-fold AUC: {roc_auc:.3f}")
print(f"Mean fold AUC: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
print("Group effects were imposed during simulation. These are not clinical validation results.")
print("Regional p-values are exploratory and unadjusted for multiple comparisons.")

In [ ]:
# Save all results
df.to_csv(OUTPUT_DIR / 'subject_data.csv', index=False)
stats_df.to_csv(OUTPUT_DIR / 'statistical_results.csv', index=False)

print("\nOutputs saved to ./outputs/")
for f in sorted(OUTPUT_DIR.glob('*')):
    print(f"  • {f.name}")
import json
with open(OUTPUT_DIR / 'metrics.json', 'w') as f:
    json.dump({'data': 'synthetic', 'seed': 42, 'subjects': len(df),
               'out_of_fold_auc': float(roc_auc), 'fold_auc': cv_scores.tolist(),
               'preprocessing': 'StandardScaler fitted within each training fold'}, f, indent=2)


---
## References

1. **Binnewijzend MAA et al. (2013).** Cerebral blood flow measured with 3D pCASL MR imaging in Alzheimer disease and MCI. *Radiology*, 267(1), 221-230.

2. **Mattsson N et al. (2014).** Association of brain amyloid-β with cerebral perfusion and structure in Alzheimer's disease and MCI. *Brain*, 137(5), 1550-1561.

3. **Mutsaerts HJMM et al. (2020).** ExploreASL: An image processing pipeline for multi-center ASL perfusion MRI studies. *NeuroImage*, 219, 117031.

4. **Park DC et al. (2025).** The Dallas Lifespan Brain Study. *Scientific Data*.

---

*Analysis demonstrates ASL-MRI methodology for preclinical AD research, relevant to the AMYPAD consortium's multimodal biomarker development program.*